# Comparing estimators

This notebook compares the estimators of the simulator (DESIGN-ESTIMATORS.md), on the same ground truth and on the same observations, in two regimes:
- **random samples:** cells drawn uniformly from the farm;
- **a robot trajectory:** the cells a random-waypoint robot observes, which are correlated, repeated, and leave large areas unvisited.

It compares their accuracy, their recall of the diseased cells, the **calibration of their confidence**, and their cost. It uses the functions of `estimator_benchmark.py` on a small configuration. The full benchmark runs as an `estimator-benchmark` exp/run (`EstimatorBenchmark-Run.ipynb`).

Results are written to a temporary results path.

In [ ]:
import pathlib
import sys
import tempfile
import warnings
sys.path.append("..")
warnings.simplefilter("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from exp_run_config import Config
from estimator_benchmark import benchmark_environment, evaluate, generate_observations
from information_model import DISEASED
from wbf_helper import create_estimator, get_geometry
Config.PROJECTNAME = "WaterBerryFarms"
workspace = pathlib.Path(tempfile.mkdtemp())
Config().set_results_path(workspace)

ESTIMATORS = ["point", "adaptive-disk", "nearest", "idw", "rbf", "gaussian-process", "gp-kriging", "gp-local",
              "gp-indicator", "occupancy", "mrf", "epidemic-pf", "cnn"]
exp_env, environment = benchmark_environment("miniberry-30", 6)
typename = exp_env["typename"]
geometry = get_geometry(typename)

def make(run):
    """A fresh estimator from its estimator exp/run"""
    return create_estimator(Config().get_experiment("estimator", run, create_data_dir=False), geometry)

def show(ax, field, title, cmap="gray", vmin=0, vmax=1):
    ax.imshow(field.T, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

**The estimators**, and what their uncertainty means (`UNCERTAINTY`). `confidence()` maps every kind to [0, 1]. The estimators with a `probability` give the probability that a cell is diseased (value < 0.75). The others are scored with a probability derived from their value, `clip(2 (1 - value))`.

The estimators that model only the diseases use inverse distance weighting for the soil moisture.

In [ ]:
rows = []
for run in ESTIMATORS:
    estimator = make(run)
    rows.append({"estimator": run, "name": estimator.name, "TYLCV estimator": type(estimator.im_tylcv).__name__,
                 "uncertainty": estimator.im_tylcv.UNCERTAINTY, "soil estimator": type(estimator.im_soil).__name__})
pd.DataFrame(rows).set_index("estimator")

## 1. The two observation regimes

The same number of observations, 100, either sampled at random or along the trajectory of a random-waypoint robot. A trajectory covers far less of the farm: its observations are adjacent, and it revisits cells.

In [ ]:
regimes = {"random": generate_observations({"kind": "random", "count": 100}, environment, typename, 0),
           "random-waypoint": generate_observations({"kind": "random-waypoint", "timesteps": 100}, environment, typename, 0)}
farm_cells = np.array([(x, y) for x in range(geometry["width"]) for y in range(geometry["height"])])
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, (regime, observations) in zip(axes, regimes.items()):
    cells = np.array([(o["x"], o["y"]) for o in observations])
    distance, _ = cKDTree(np.unique(cells, axis=0)).query(farm_cells)
    show(ax, environment.tylcv.value, "")
    ax.plot(cells[:, 0], cells[:, 1], ".", color="tab:red" if regime == "random" else "tab:blue")
    ax.set_title(f"{regime}: {len(np.unique(cells, axis=0))} distinct cells, "
                 f"{np.mean(distance <= 2):.0%} of the farm within 2 cells", fontsize=9)
plt.suptitle("TYLCV ground truth (white healthy, gray infected, black destroyed) and the observations")
plt.show()

## 2. The estimates

For a selection of the estimators, the TYLCV estimate after the 100 observations of each regime. Each row shows:
- **value:** the estimated value;
- **error:** the absolute error (white is no error, red is 0.5 or more);
- **confidence:** white is certain, black ignorant;
- **probability:** the probability of disease, for the estimators that provide it.

The maps show which estimators extrapolate, which stay local, and where each one is confident.

In [ ]:
SELECTION = ["point", "adaptive-disk", "idw", "gaussian-process", "gp-kriging", "gp-indicator", "occupancy", "mrf", "epidemic-pf", "cnn"]
for regime, observations in regimes.items():
    fig, axes = plt.subplots(len(SELECTION), 4, figsize=(8, 2.1 * len(SELECTION)))
    for row, run in zip(axes, SELECTION):
        estimator = make(run)
        for observation in observations:
            estimator.add_observation(observation)
        estimator.proceed(1)
        im = estimator.im_tylcv
        show(row[0], im.value, f"{run}: value")
        show(row[1], np.abs(im.value - environment.tylcv.value), "error", cmap="Reds", vmax=0.5)
        show(row[2], im.confidence(), "confidence")
        if im.probability is not None:
            show(row[3], im.probability, "probability of disease", cmap="Reds")
        else:
            row[3].axis("off")
    fig.suptitle(f"TYLCV estimates, {regime}, 100 observations", y=1.0)
    plt.tight_layout()
    plt.show()

## 3. Accuracy against the number of observations

Every estimator receives the identical observation sequences: 2 repetitions of each regime, up to 300 observations. The estimators are evaluated on the tomato cells (TYLCV) at the checkpoints. An estimator is stopped once a single estimate takes longer than 10 seconds.

In [ ]:
CHECKPOINTS = [10, 20, 50, 100, 200, 300]
SAMPLERS = {"random": {"kind": "random", "count": 300}, "random-waypoint": {"kind": "random-waypoint", "timesteps": 300}}
rows = []
for regime, sampler in SAMPLERS.items():
    for repetition in range(2):
        observations = generate_observations(sampler, environment, typename, repetition)
        for run in ESTIMATORS:
            for row in evaluate(make(run), environment, observations, CHECKPOINTS, stop_at=10.0):
                rows.append({"sampler": regime, "repetition": repetition, "estimator": run} | row)
results = pd.DataFrame(rows)
tylcv = results[results.field == "tylcv"].groupby(["sampler", "estimator", "observations", "metric"]).value.mean().unstack("metric")

In [ ]:
metrics = [("mae", "MAE"), ("asymmetric-error", "asymmetric error (missed disease x10)"), ("recall", "recall of the diseased cells")]
fig, axes = plt.subplots(len(metrics), 2, figsize=(12, 3.6 * len(metrics)), sharex=True)
for i, (metric, label) in enumerate(metrics):
    for j, regime in enumerate(SAMPLERS):
        ax = axes[i, j]
        for k, run in enumerate(ESTIMATORS):
            curve = tylcv.loc[(regime, run)][metric]
            ax.plot(curve.index, curve.values, marker=".", label=run, color=plt.cm.tab20(k),
                    linestyle="--" if run in ["point", "adaptive-disk", "gaussian-process"] else "-")
        ax.set_xscale("log")
        ax.set_title(f"{regime}: {label}", fontsize=10)
        ax.set_xlabel("observations")
axes[0, 1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 4. Confidence

**Calibration.** For the estimators with a probability of disease, the reliability diagram compares the predicted probability with the observed frequency of disease. It uses all the tomato cells after 300 observations, pooled over the two repetitions. A calibrated estimator lies on the diagonal; below the diagonal it is overconfident about disease, above it it misses disease.

In [ ]:
PROBABILISTIC = [run for run in ESTIMATORS if make(run).im_tylcv.UNCERTAINTY == "probability"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
mask = environment.my_tomato_mask
for ax, (regime, sampler) in zip(axes, SAMPLERS.items()):
    for run in PROBABILISTIC:
        probabilities, diseased = [], []
        for repetition in range(2):
            estimator = make(run)
            for observation in generate_observations(sampler, environment, typename, repetition):
                estimator.add_observation(observation)
            estimator.proceed(1)
            probabilities.append(estimator.im_tylcv.probability[mask])
            diseased.append(environment.tylcv.value[mask] < DISEASED)
        p, d = np.concatenate(probabilities), np.concatenate(diseased)
        bins = np.minimum((p * 10).astype(int), 9)
        centers = [p[bins == b].mean() for b in range(10) if np.any(bins == b)]
        frequencies = [d[bins == b].mean() for b in range(10) if np.any(bins == b)]
        ax.plot(centers, frequencies, marker="o", label=run)
    ax.plot([0, 1], [0, 1], color="black", linewidth=0.8)
    ax.set_xlabel("predicted probability of disease"); ax.set_ylabel("observed frequency")
    ax.set_title(f"{regime}, 300 observations")
axes[0].legend()
plt.show()

**Calibration metrics and usefulness of the uncertainty,** after 300 observations, as the mean over the repetitions:
- **brier, nll, ece:** the Brier score, the negative log-likelihood, and the expected calibration error of the probability of disease (native or derived). Lower is better.
- **confident-errors:** the fraction of cells where the true class got a probability below 0.1.
- **uncertainty-error-correlation:** the rank correlation between `1 - confidence()` and the absolute error, i.e. whether the uncertainty points at the errors. This is what a confidence-guided policy needs.
- **coverage-1sd / coverage-2sd:** for the estimators with a standard deviation, the fraction of cells within 1 / 2 standard deviations. A calibrated Gaussian would cover 68% / 95%.

In [ ]:
final = tylcv.xs(300, level="observations")
columns = ["brier", "nll", "ece", "confident-errors", "uncertainty-error-correlation", "coverage-1sd", "coverage-2sd", "native-probability"]
final[columns].round(3)

## 5. Cost

The time of one estimate, against the number of observations. The adaptive disk, the occupancy grid and the CNN are almost constant, while the GP grows quickly. Kriging and the local GP bound it with `gp-max-observations` and tiles.

In [ ]:
cost = results[results.metric == "proceed-seconds"].groupby(["estimator", "observations"]).value.mean()
fig, ax = plt.subplots(figsize=(7, 4.5))
for k, run in enumerate(ESTIMATORS):
    ax.plot(cost.loc[run].index, cost.loc[run].values, marker=".", label=run, color=plt.cm.tab20(k))
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("observations"); ax.set_ylabel("seconds per estimate")
ax.legend(fontsize=7, ncol=2)
plt.show()

## 6. Summary

The main metrics after 300 observations, per regime:

In [ ]:
summary = final[["mae", "asymmetric-error", "recall", "auc", "brier", "uncertainty-error-correlation"]].unstack("sampler")
summary.round(3)

**Observations on this configuration** (Miniberry-30 at day 6, where the disease covers few cells; 2 repetitions; see the full benchmark for more scenarios):

- **The regime changes everything.**
  - **Random samples:** most estimators find the diseased cells, with a recall up to 1.0 after 300 observations.
  - **A trajectory:** with the same number of observations, it leaves most of the disease unvisited, and the recall is at most about 0.4.
  - **Rankings:** an estimator's ranking under random sampling does not carry over to trajectories, so estimators for IPP should be evaluated on trajectories.
- **The original GP configuration fails on trajectories.** `gaussian-process` with `gp-normalize-y: false` has a prior mean of 0 ("destroyed"), so it predicts disease wherever the robot has not been. That gives it a high recall but a large error. `gp-kriging` (normalized, bounded) fixes it.
- **Model-based and learned estimators are the most accurate and the best calibrated.**
  - **`epidemic-pf` and `cnn`** have the lowest asymmetric error and Brier score in both regimes.
  - **The particle filter is optimistic here:** it assumes the environment's true epidemic parameters, and only the seeds differ. The benchmark can test mismatched parameters (`pf-*` in the estimator exp/run).
  - **The CNN** was trained on other environments with the same models.
- **Only some estimators know where they are wrong.** The rank correlation between their uncertainty and their error is close to 1 for the probability-based estimators (`epidemic-pf`, `cnn`, `mrf`, `occupancy`, `gp-indicator`), and around 0.1 for the coverage and distance heuristics (`point`, `adaptive-disk`, `nearest`, `idw`, `rbf`). A confidence-guided policy should use the former.
- **The GP standard deviations are too wide:** about 90% of the cells are within one standard deviation, where a calibrated Gaussian would have 68%. The binary disease patches violate the GP's smoothness assumptions.
- **Smoothing models can miss small patches.** The `mrf` and `occupancy` estimators stay close to their prior of 0.1. Their probabilities under-predict disease (above the diagonal of the reliability diagram), and their recall is low.
- **Cost:** the GP grows with the observations. Kriging and the local GP stay bounded, and the disk, occupancy and CNN estimators are nearly constant.